In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
df = spark.read.format('parquet').load('abfss://bronze@databrickse2esa.dfs.core.windows.net/customers')
df = df.withColumn('domain', split(col('email'), '@')[1]).withColumn('fullname', concat(col('first_name'), col('last_name'))).drop('_rescued_data','first_name','last_name')
display(df)

In [0]:
newdf = df.groupBy('domain').agg(count(col('customer_id')).alias('Total_customers')).orderBy(col('Total_customers').desc())
display(newdf)

In [0]:
df.write.format('delta').mode('overwrite').save('abfss://silver@databrickse2esa.dfs.core.windows.net/customers')

In [0]:
%sql
create table if not exists databricks_cata.silver.customers_silver
using delta
location 'abfss://silver@databrickse2esa.dfs.core.windows.net/customers';

select * from databricks_cata.silver.customers_silver;